In [1]:
import plotly.graph_objects as go

import os
os.environ["SHAPE_RESTORE_SHX"] = "YES"

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.features import geometry_mask
from pathlib import Path
import re
import gc


import winsound

In [2]:

mspa_classes = {
    1: "Core", 2: "Small Islet", 3: "Perforation", 4: "Edge", 0: "No Forest", 6: "Medium Islet", 7: "Big Islet",
}

class_colors = {
    "Core": "#006400",
    "Small Islet": "#FFA07A",
    "Perforation": "#905590",
    "Edge": "#90EE90",
    "No Forest": "#B3B3B3",
    "Medium Islet": "#F0E68C" ,
    "Big Islet":  "#808000",
}

# Sankey Plot - Argentina

In [3]:
folder = Path(r"C:\Users\")
shapefile_path = Path(r"C:\Users\Country_areas\APAF_Argentina_4326.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [1, 2, 3, 4, 6, 7]

years = [1986, 1992, 1997, 1999, 2010, 2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"mspa_500mt_v500perforation_vIslets_Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

# Sentinel code for "disappeared" pixels: nodata, background, or outside the
# AOI at the END year of a given hop. Not a real raster value — only appears
# in to_val, never in from_val (a pixel must be valid to START a hop).
DISAPPEARED_CODE = 254
DISAPPEARED_LABEL = "Deforested"

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask of pixels OUTSIDE the shapefile geometry — same for every year
        # (the AOI doesn't change), so computed once per block.
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # Per-year validity: True where that year's pixel is usable (inside
        # AOI, not nodata, not background=0). Computed separately per band,
        # instead of collapsing across all years with .any(axis=0) — that
        # collapse is what was silently dropping pixels that disappear partway
        # through the series from every hop, including the ones before they
        # disappeared.
        valid_year = np.ones(stack.shape, dtype=bool)
        valid_year &= ~outside_mask[np.newaxis, :, :]
        valid_year &= (stack != 0)
        if ref_nodata is not None:
            valid_year &= (stack != ref_nodata)

        for k in range(MAX_HOPS):
            from_val = stack[k]
            to_val_raw = stack[k + 1]

            # A pixel must be valid at the START of this hop to be counted.
            # It does NOT need to still be valid at the end — if it isn't,
            # it's routed to the "Disappeared" sentinel instead of being
            # dropped, so the transition still shows up in the diagram.
            mask = valid_year[k]

            if not mask.any():
                continue

            to_val = np.where(valid_year[k + 1], to_val_raw, DISAPPEARED_CODE)

            pairs, counts = np.unique(
                np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
            )
            for (fv, tv), c in zip(pairs, counts):
                key = (k, int(fv), int(tv))
                transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, valid_year
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mspa_classes.get(from_val, f"Unknown ({from_val})")
    to_name = DISAPPEARED_LABEL if to_val == DISAPPEARED_CODE else mspa_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
#traj_df.to_csv(folder / "MSPA_trajectory_Argentina_test.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.1f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.1f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.1f}K"
    else:
        return f"{sign}{n:.1f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)} - {format_short(node_totals[node])} ha" for node in all_nodes]

# "Disappeared" gets its own color since it's not in class_colors; edit
# DISAPPEARED_COLOR to whatever fits your palette.
DISAPPEARED_COLOR = "#A6A6A6"
node_colors = [
    DISAPPEARED_COLOR if node_class(n) == DISAPPEARED_LABEL else class_colors.get(node_class(n), "#999999")
    for n in all_nodes
]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [
    hex_to_rgba(DISAPPEARED_COLOR if cn == DISAPPEARED_LABEL else class_colors.get(cn, "#999999"))
    for cn in traj_df["source_class"]
]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"MSPA trajectory for legislative milestones ({years}) - Argentina",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_MSPA_Argentina_laws_vx.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_mspa_Argentina_laws.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")

Using years: [1986, 1992, 1997, 1999, 2010, 2023]
Collected 207 unique (hop, from, to) transitions
    hop source_class target_class              source              target  \
0     0         Core         Core         Core (1986)         Core (1992)   
1     0         Core  Perforation         Core (1986)  Perforation (1992)   
2     0         Edge         Core         Edge (1986)         Core (1992)   
3     0         Edge  Perforation         Edge (1986)  Perforation (1992)   
4     0         Edge         Edge         Edge (1986)         Edge (1992)   
5     0         Edge   Deforested         Edge (1986)   Deforested (1992)   
6     1         Core         Core         Core (1992)         Core (1997)   
7     1         Core  Perforation         Core (1992)  Perforation (1997)   
8     1         Core         Edge         Core (1992)         Edge (1997)   
9     1         Core   Deforested         Core (1992)   Deforested (1997)   
10    1  Perforation  Perforation  Perforation (1992) 

# Sankey Plot - Brazil

In [ ]:
folder = Path(r"C:\Users\")
shapefile_path = Path(r"C:\Users\Country_areas\APAF_Brazil_4326.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [1, 2, 3, 4, 6, 7]

years = [1986, 2000, 2008, 2012, 2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"mspa_500mt_v500perforation_vIslets_Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

# Sentinel code for "disappeared" pixels: nodata, background, or outside the
# AOI at the END year of a given hop. Not a real raster value — only appears
# in to_val, never in from_val (a pixel must be valid to START a hop).
DISAPPEARED_CODE = 254
DISAPPEARED_LABEL = "Deforested"

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask of pixels OUTSIDE the shapefile geometry — same for every year
        # (the AOI doesn't change), so computed once per block.
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # Per-year validity: True where that year's pixel is usable (inside
        # AOI, not nodata, not background=0). Computed separately per band,
        # instead of collapsing across all years with .any(axis=0) — that
        # collapse is what was silently dropping pixels that disappear partway
        # through the series from every hop, including the ones before they
        # disappeared.
        valid_year = np.ones(stack.shape, dtype=bool)
        valid_year &= ~outside_mask[np.newaxis, :, :]
        valid_year &= (stack != 0)
        if ref_nodata is not None:
            valid_year &= (stack != ref_nodata)

        for k in range(MAX_HOPS):
            from_val = stack[k]
            to_val_raw = stack[k + 1]

            # A pixel must be valid at the START of this hop to be counted.
            # It does NOT need to still be valid at the end — if it isn't,
            # it's routed to the "Disappeared" sentinel instead of being
            # dropped, so the transition still shows up in the diagram.
            mask = valid_year[k]

            if not mask.any():
                continue

            to_val = np.where(valid_year[k + 1], to_val_raw, DISAPPEARED_CODE)

            pairs, counts = np.unique(
                np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
            )
            for (fv, tv), c in zip(pairs, counts):
                key = (k, int(fv), int(tv))
                transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, valid_year
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mspa_classes.get(from_val, f"Unknown ({from_val})")
    to_name = DISAPPEARED_LABEL if to_val == DISAPPEARED_CODE else mspa_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
traj_df.to_csv(folder / "MSPA_trajectory_Brazil_laws_test.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.1f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.1f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.1f}K"
    else:
        return f"{sign}{n:.1f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)} - {format_short(node_totals[node])} ha" for node in all_nodes]

# "Disappeared" gets its own color since it's not in class_colors; edit
# DISAPPEARED_COLOR to whatever fits your palette.
DISAPPEARED_COLOR = "#A6A6A6"
node_colors = [
    DISAPPEARED_COLOR if node_class(n) == DISAPPEARED_LABEL else class_colors.get(node_class(n), "#999999")
    for n in all_nodes
]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [
    hex_to_rgba(DISAPPEARED_COLOR if cn == DISAPPEARED_LABEL else class_colors.get(cn, "#999999"))
    for cn in traj_df["source_class"]
]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"MSPA trajectory for legislative milestones ({years}) - Brazil",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_MSPA_Brazil_laws_vx.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_mspa_Brazil_laws.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")

# Sankey Plot - Paraguay

In [ ]:
folder = Path(r"C:\Users\")
shapefile_path = Path(r"C:\Users\Country_areas\APAF_Paraguay_4326.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [1, 2, 3, 4, 6, 7]

years = [1986, 2004, 2006, 2008, 2013, 2018, 2020, 2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"mspa_500mt_v500perforation_vIslets_Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

# Sentinel code for "disappeared" pixels: nodata, background, or outside the
# AOI at the END year of a given hop. Not a real raster value — only appears
# in to_val, never in from_val (a pixel must be valid to START a hop).
DISAPPEARED_CODE = 254
DISAPPEARED_LABEL = "Deforested"

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask of pixels OUTSIDE the shapefile geometry — same for every year
        # (the AOI doesn't change), so computed once per block.
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # Per-year validity: True where that year's pixel is usable (inside
        # AOI, not nodata, not background=0). Computed separately per band,
        # instead of collapsing across all years with .any(axis=0) — that
        # collapse is what was silently dropping pixels that disappear partway
        # through the series from every hop, including the ones before they
        # disappeared.
        valid_year = np.ones(stack.shape, dtype=bool)
        valid_year &= ~outside_mask[np.newaxis, :, :]
        valid_year &= (stack != 0)
        if ref_nodata is not None:
            valid_year &= (stack != ref_nodata)

        for k in range(MAX_HOPS):
            from_val = stack[k]
            to_val_raw = stack[k + 1]

            # A pixel must be valid at the START of this hop to be counted.
            # It does NOT need to still be valid at the end — if it isn't,
            # it's routed to the "Disappeared" sentinel instead of being
            # dropped, so the transition still shows up in the diagram.
            mask = valid_year[k]

            if not mask.any():
                continue

            to_val = np.where(valid_year[k + 1], to_val_raw, DISAPPEARED_CODE)

            pairs, counts = np.unique(
                np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
            )
            for (fv, tv), c in zip(pairs, counts):
                key = (k, int(fv), int(tv))
                transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, valid_year
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mspa_classes.get(from_val, f"Unknown ({from_val})")
    to_name = DISAPPEARED_LABEL if to_val == DISAPPEARED_CODE else mspa_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
traj_df.to_csv(folder / "MSPA_trajectory_Paraguay_laws_test.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.1f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.1f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.1f}K"
    else:
        return f"{sign}{n:.1f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)} - {format_short(node_totals[node])} ha" for node in all_nodes]

# "Disappeared" gets its own color since it's not in class_colors; edit
# DISAPPEARED_COLOR to whatever fits your palette.
DISAPPEARED_COLOR = "#A6A6A6"
node_colors = [
    DISAPPEARED_COLOR if node_class(n) == DISAPPEARED_LABEL else class_colors.get(node_class(n), "#999999")
    for n in all_nodes
]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [
    hex_to_rgba(DISAPPEARED_COLOR if cn == DISAPPEARED_LABEL else class_colors.get(cn, "#999999"))
    for cn in traj_df["source_class"]
]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"MSPA trajectory for legislative milestones ({years}) - Paraguay",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_MSPA_Paraguay_laws_vx.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_mspa_Paraguay_laws.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")

# Sankey LULC - Laws

In [3]:
mapbiomas_classes = {
    0: "nan", 2: "Other nat. veg.", 3: "Native Forest", 9: "Forest Plantation",
    11: "Wetland", 15: "Pasture", 19: "Temporary Crops", 22: "Non veg. area",
    27: "Not Observed", 33: "Water", 36: "Perennial Crops", 39: "Temporary Crops",
}

class_colors = {
    "Native Forest": "#1a9850",
    "Pasture": "#FFD700",
    "Temporary Crops": "#8e7cc3",
    "Other nat. veg.": "#7dc975",
    "Forest Plantation": "#A9A9A9",
    "Wetland": "#519799",
    "Non veg. area": "#708090",
    "Not Observed": "#DCDCDC",
    "Water": "#2532e4",
    "Perennial Crops": "#f3b4f1",
}

# Argentina

In [5]:
folder = Path(r"C:\Users\")
shapefile_path = Path(r"C:\Users\Country_areas\APAF_Argentina_4326.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [3, 15, 19]

years = [1986, 1992, 1997, 1999, 2010, 2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask pixels outside the shapefile geometry
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # A pixel is excluded entirely if it's nodata/outside-AOI in ANY year,
        # or if its first-year value isn't one of the classes we start trajectories from.
        nodata_any = np.zeros(stack.shape[1:], dtype=bool)
        if ref_nodata is not None:
            nodata_any |= (stack == ref_nodata).any(axis=0)
        nodata_any |= (stack == 0).any(axis=0)
        nodata_any |= outside_mask

        valid_start = np.isin(stack[0], INITIAL_CLASSES) & ~nodata_any

        if valid_start.any():
            # Direct per-year transitions: for each consecutive pair of years,
            # use the ACTUAL value at each year (no compressed "change event"
            # indexing). This means every valid pixel contributes a transition
            # at every hop, whether or not its class actually changed that
            # hop — a pixel that stays the same from one year to the next
            # still shows up as ClassX (year1) -> ClassX (year2), instead of
            # being dropped from the diagram just because nothing changed.
            for k in range(MAX_HOPS):
                from_val = stack[k]
                to_val = stack[k + 1]
                mask = valid_start

                if not mask.any():
                    continue

                pairs, counts = np.unique(
                    np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
                )
                for (fv, tv), c in zip(pairs, counts):
                    key = (k, int(fv), int(tv))
                    transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, nodata_any, valid_start
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mapbiomas_classes.get(from_val, f"Unknown ({from_val})")
    to_name = mapbiomas_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
#traj_df.to_csv(folder / "forest_pasture_tempcrops_chained_trajectory_Argentina.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.2f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.2f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.2f}K"
    else:
        return f"{sign}{n:.2f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)}<br>{format_short(node_totals[node])} ha" for node in all_nodes]

node_colors = [class_colors.get(node_class(n), "#999999") for n in all_nodes]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [hex_to_rgba(class_colors.get(cn, "#999999")) for cn in traj_df["source_class"]]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=30,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"Chained trajectory - Laws - Argentina",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_chained_trajectory_Argentina_laws_vx.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_chained_trajectory_Argentina.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")

Using years: [1986, 1992, 1997, 1999, 2010, 2023]
Collected 293 unique (hop, from, to) transitions
    hop       source_class       target_class                    source  \
0     0      Native Forest      Native Forest      Native Forest (1986)   
1     0      Native Forest    Perennial Crops      Native Forest (1986)   
2     1      Native Forest      Native Forest      Native Forest (1992)   
3     1      Native Forest  Forest Plantation      Native Forest (1992)   
4     1      Native Forest    Perennial Crops      Native Forest (1992)   
5     1    Perennial Crops    Perennial Crops    Perennial Crops (1992)   
6     2      Native Forest      Native Forest      Native Forest (1997)   
7     2      Native Forest    Perennial Crops      Native Forest (1997)   
8     2  Forest Plantation  Forest Plantation  Forest Plantation (1997)   
9     2    Perennial Crops      Native Forest    Perennial Crops (1997)   
10    2    Perennial Crops    Perennial Crops    Perennial Crops (1997)   
1

# Brazil

In [ ]:
folder = Path(r"C:\Users")
shapefile_path = Path(r"C:\Users\Country_areas\APAF_Brazil_4326.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [3, 15, 19]

years = [1986, 2000, 2008, 2012, 2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask pixels outside the shapefile geometry
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # A pixel is excluded entirely if it's nodata/outside-AOI in ANY year,
        # or if its first-year value isn't one of the classes we start trajectories from.
        nodata_any = np.zeros(stack.shape[1:], dtype=bool)
        if ref_nodata is not None:
            nodata_any |= (stack == ref_nodata).any(axis=0)
        nodata_any |= (stack == 0).any(axis=0)
        nodata_any |= outside_mask

        valid_start = np.isin(stack[0], INITIAL_CLASSES) & ~nodata_any

        if valid_start.any():
            # Direct per-year transitions: for each consecutive pair of years,
            # use the ACTUAL value at each year (no compressed "change event"
            # indexing). This means every valid pixel contributes a transition
            # at every hop, whether or not its class actually changed that
            # hop — a pixel that stays the same from one year to the next
            # still shows up as ClassX (year1) -> ClassX (year2), instead of
            # being dropped from the diagram just because nothing changed.
            for k in range(MAX_HOPS):
                from_val = stack[k]
                to_val = stack[k + 1]
                mask = valid_start

                if not mask.any():
                    continue

                pairs, counts = np.unique(
                    np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
                )
                for (fv, tv), c in zip(pairs, counts):
                    key = (k, int(fv), int(tv))
                    transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, nodata_any, valid_start
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mapbiomas_classes.get(from_val, f"Unknown ({from_val})")
    to_name = mapbiomas_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
#traj_df.to_csv(folder / "forest_pasture_tempcrops_chained_trajectory_Argentina.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.2f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.2f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.2f}K"
    else:
        return f"{sign}{n:.2f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)}<br>{format_short(node_totals[node])} ha" for node in all_nodes]

node_colors = [class_colors.get(node_class(n), "#999999") for n in all_nodes]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [hex_to_rgba(class_colors.get(cn, "#999999")) for cn in traj_df["source_class"]]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=30,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"Chained trajectory - Laws - Brazil",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_chained_trajectory_Brazil_laws_vx.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_chained_trajectory_Brazil.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")

Using years: [1986, 2000, 2008, 2012, 2023]


# Paraguay

In [ ]:
folder = Path(r"C:\Users")
shapefile_path = Path(r"C:\Users\Country_areasAPAF_Paraguay_4326.shp")

PIXEL_AREA_HA = 0.079956
INITIAL_CLASSES = [3, 15, 19]

years = [1986, 2004, 2006, 2008, 2013, 2018, 2020, 2023]
MAX_HOPS = len(years) - 1
print(f"Using years: {years}")

tif_paths = [folder / f"Reclass_img_{y}_v0806.tif" for y in years]
missing = [p for p in tif_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

aoi = gpd.read_file(shapefile_path)

with rasterio.open(tif_paths[0]) as ref:
    ref_nodata = ref.nodata
    ref_crs = ref.crs

    aoi_reproj = aoi.to_crs(ref_crs)
    geometries = [geom for geom in aoi_reproj.geometry if geom is not None]

    minx, miny, maxx, maxy = aoi_reproj.total_bounds
    aoi_window = from_bounds(minx, miny, maxx, maxy, transform=ref.transform)
    aoi_window = aoi_window.intersection(Window(0, 0, ref.width, ref.height))
    aoi_window = aoi_window.round_offsets().round_lengths()

    block_windows = list(ref.block_windows(1))

open_srcs = [rasterio.open(p) for p in tif_paths]

transition_counts = {}

try:
    for _, window in block_windows:
        try:
            block_window = window.intersection(aoi_window)
        except rasterio.errors.WindowError:
            continue
        if block_window.width <= 0 or block_window.height <= 0:
            continue

        blocks = [src.read(1, window=block_window) for src in open_srcs]
        stack = np.stack(blocks, axis=0).astype(np.uint8)
        del blocks

        stack[stack == 39] = 19

        # Mask pixels outside the shapefile geometry
        block_transform = open_srcs[0].window_transform(block_window)
        outside_mask = geometry_mask(
            geometries, out_shape=stack.shape[1:], transform=block_transform, invert=False
        )

        # A pixel is excluded entirely if it's nodata/outside-AOI in ANY year,
        # or if its first-year value isn't one of the classes we start trajectories from.
        nodata_any = np.zeros(stack.shape[1:], dtype=bool)
        if ref_nodata is not None:
            nodata_any |= (stack == ref_nodata).any(axis=0)
        nodata_any |= (stack == 0).any(axis=0)
        nodata_any |= outside_mask

        valid_start = np.isin(stack[0], INITIAL_CLASSES) & ~nodata_any

        if valid_start.any():
            # Direct per-year transitions: for each consecutive pair of years,
            # use the ACTUAL value at each year (no compressed "change event"
            # indexing). This means every valid pixel contributes a transition
            # at every hop, whether or not its class actually changed that
            # hop — a pixel that stays the same from one year to the next
            # still shows up as ClassX (year1) -> ClassX (year2), instead of
            # being dropped from the diagram just because nothing changed.
            for k in range(MAX_HOPS):
                from_val = stack[k]
                to_val = stack[k + 1]
                mask = valid_start

                if not mask.any():
                    continue

                pairs, counts = np.unique(
                    np.stack([from_val[mask], to_val[mask]], axis=1), axis=0, return_counts=True
                )
                for (fv, tv), c in zip(pairs, counts):
                    key = (k, int(fv), int(tv))
                    transition_counts[key] = transition_counts.get(key, 0) + int(c)

        del stack, outside_mask, nodata_any, valid_start
        gc.collect()

finally:
    for src in open_srcs:
        src.close()

print(f"Collected {len(transition_counts)} unique (hop, from, to) transitions")

records = []
for (hop, from_val, to_val), count in transition_counts.items():
    from_name = mapbiomas_classes.get(from_val, f"Unknown ({from_val})")
    to_name = mapbiomas_classes.get(to_val, f"Unknown ({to_val})")
    records.append({
        "hop": hop,
        "source_class": from_name,
        "target_class": to_name,
        "source": f"{from_name} ({years[hop]})",
        "target": f"{to_name} ({years[hop + 1]})",
        "count": count,
        "area_ha": count * PIXEL_AREA_HA,
    })

traj_df = pd.DataFrame(records)
#traj_df.to_csv(folder / "forest_pasture_tempcrops_chained_trajectory_Argentina.csv", index=False)
print(traj_df.head(20))

# --- Area that changed per hop (source_class != target_class) ---
changed_per_hop = (
    traj_df[traj_df["source_class"] != traj_df["target_class"]]
    .groupby("hop")["area_ha"].sum()
)
print("Area changed per hop (ha):")
print(changed_per_hop)

# --- Helper: shorten large numbers (e.g. 3,245,120 -> "3.2M") ---
def format_short(n):
    n = float(n)
    sign = "-" if n < 0 else ""
    n = abs(n)
    if n >= 1_000_000_000:
        return f"{sign}{n/1_000_000_000:.2f}B"
    elif n >= 1_000_000:
        return f"{sign}{n/1_000_000:.2f}M"
    elif n >= 1_000:
        return f"{sign}{n/1_000:.2f}K"
    else:
        return f"{sign}{n:.2f}"

# --- Build Sankey ---
all_nodes = pd.unique(traj_df[["source", "target"]].values.ravel())
node_index = {name: i for i, name in enumerate(all_nodes)}

# --- Compute total area (ha) represented by each node (class, year) ---
# For nodes that appear as a "source" (all years except the last), total = sum of outgoing flows.
# For nodes that only appear as a "target" (the last year), total = sum of incoming flows.
source_totals = traj_df.groupby("source")["area_ha"].sum()
target_totals = traj_df.groupby("target")["area_ha"].sum()

node_totals = {}
for node in all_nodes:
    if node in source_totals.index:
        node_totals[node] = source_totals[node]
    elif node in target_totals.index:
        node_totals[node] = target_totals[node]
    else:
        node_totals[node] = 0

def node_class(label):
    return re.sub(r"\s*\(\d{4}\)", "", label)

# Node labels now show only the class name + shortened area (no year — years are
# shown once per column via annotations instead).
node_labels = [f"{node_class(node)}<br>{format_short(node_totals[node])} ha" for node in all_nodes]

node_colors = [class_colors.get(node_class(n), "#999999") for n in all_nodes]
source_idx = traj_df["source"].map(node_index)
target_idx = traj_df["target"].map(node_index)
values = traj_df["area_ha"]

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

link_colors = [hex_to_rgba(class_colors.get(cn, "#999999")) for cn in traj_df["source_class"]]

fig = go.Figure(data=[go.Sankey(textfont=dict(size=14, color="black"),
    node=dict(
        pad=30,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values,
        color=link_colors,
    )
)])

fig.update_layout(
    title_text=f"Chained trajectory - Laws - Paraguay",
    font_size=11,
    width=1300,
    height=800,
    margin=dict(t=150),
)

# --- Year headers: one label per column, at the top of the chain ---
n_years = len(years)
x_positions = [i / (n_years - 1) for i in range(n_years)]

for x, year in zip(x_positions, years):
    fig.add_annotation(
        x=x,
        y=-0.06,
        xref="paper",
        yref="paper",
        text=f"<b>{year}</b>",
        showarrow=False,
        font=dict(size=16, color="black"),
        align="center",
    )

# --- Annotations: area changed between consecutive years (shortened) ---
for hop in range(MAX_HOPS):
    changed_ha = changed_per_hop.get(hop, 0)
    changed_ha_str = format_short(changed_ha)
    x_mid = (x_positions[hop] + x_positions[hop + 1]) / 2
    annotation_text = f"{changed_ha_str} ha changed"
    fig.add_annotation(
        x=x_mid,
        y=-0.1,
        xref="paper",
        yref="paper",
        text=annotation_text,
        showarrow=False,
        font=dict(size=12, color="black"),
        align="center",
    )

out_html = folder / "sankey_chained_trajectory_Paraguay_laws_vx.html"
fig.write_html(out_html)
try:
    fig.write_image(folder / "sankey_chained_trajectory_Paraguay.png")
except Exception as e:
    print(f"Could not save PNG (install kaleido for PNG export): {e}")

print(f"Saved: {out_html}")